# CASMI26 — learned-ranker production baseline

This notebook runs the best-known Kaggle architecture (score 0.139):

1. multispectrum molecular-formula hypotheses from the frozen training/COCONUT formula index
2. formula-specific PubChem candidates
3. spectral-neighbor pruning and fingerprint features
4. learned LightGBM ranking
5. same-polarity retrieval fallback for molecules without a positive ranking
6. build and validate `submission.csv`

Experimental generated-formula and visible-test-specific PubChem-expansion code is intentionally excluded from production inference.

In [1]:
from pathlib import Path
import shutil
import subprocess
import sys

ASSET_ROOT = Path(
    "/kaggle/input/datasets/colleendavis/casmi26-inference-assets"
)
ASSET_PROJECT = ASSET_ROOT / "project"
MIST_ROOT = ASSET_ROOT / "MIST"
WHEELHOUSE = ASSET_ROOT / "wheelhouse"

COMPETITION_ROOT = Path(
    "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra"
)
TEST_PATH = COMPETITION_ROOT / "test.parquet"
SAMPLE_SUBMISSION_PATH = COMPETITION_ROOT / "sample_submission.csv"

PROJECT_ROOT = Path("/kaggle/working/casmi26-project")

HDF_FORMULAS_PER_BATCH = 8
HDF_WORKERS = 2

print("Python:", sys.version)
print("Asset root:", ASSET_ROOT)
print("Competition root:", COMPETITION_ROOT)
print("Working project:", PROJECT_ROOT)

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Asset root: /kaggle/input/datasets/colleendavis/casmi26-inference-assets
Competition root: /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra
Working project: /kaggle/working/casmi26-project


## 1. Verify the attached Kaggle dataset

Do not continue if this cell fails. It verifies that the notebook is using the new guard-5 asset version and that the offline RDKit wheel is actually present.

In [2]:
required = [
    ASSET_PROJECT / "scripts/run_kaggle_guard5_inference.py",
    ASSET_PROJECT / "scripts/build_test_multispectrum_formula_hypotheses.py",
    ASSET_PROJECT / "scripts/prepare_test_guard5_candidates.py",
    ASSET_PROJECT / "scripts/score_test_guard5_spectrum_fingerprint.py",
    ASSET_PROJECT / "scripts/build_test_learned_ranker_features.py",
    ASSET_PROJECT / "scripts/rank_test_learned_ranker.py",
    ASSET_PROJECT / "scripts/rank_test_negative_retrieval.py",
    ASSET_PROJECT / "scripts/build_submission.py",
    ASSET_PROJECT / "src",
    ASSET_PROJECT / "data/processed/formula_index",
    ASSET_PROJECT / "data/processed/structure_index",
    ASSET_PROJECT / "data/processed/retrieval_index",
    ASSET_PROJECT / "data/processed/pubchem_formula_index",
    (
        ASSET_PROJECT
        / "data/processed/spectrum_fingerprint/prototype/"
          "spectrum_to_morgan2048.pt"
    ),
    (
        ASSET_PROJECT
        / "data/processed/diagnostics/large_validation_audit/"
          "true_formula_neighbor_benchmark/"
          "learned_ranker_top500/lgbm_ranker.txt"
    ),
    ASSET_PROJECT / "data/external/coconut",
    WHEELHOUSE,
    TEST_PATH,
    SAMPLE_SUBMISSION_PATH,
]

missing = [path for path in required if not path.exists()]

if missing:
    raise FileNotFoundError(
        "The attached Kaggle inputs are incomplete or stale:\n"
        + "\n".join(str(path) for path in missing)
    )

rdkit_wheels = sorted(WHEELHOUSE.glob("rdkit-*-cp313-*.whl"))
lightgbm_wheels = sorted(WHEELHOUSE.glob("lightgbm-*.whl"))

if len(rdkit_wheels) != 1:
    raise RuntimeError(
        f"Expected one CPython 3.13 RDKit wheel; found {rdkit_wheels}"
    )

if len(lightgbm_wheels) != 1:
    raise RuntimeError(
        f"Expected one LightGBM wheel; found {lightgbm_wheels}"
    )

orchestrator = ASSET_PROJECT / "scripts/run_kaggle_guard5_inference.py"
orchestrator_text = orchestrator.read_text()

required_markers = [
    "build_test_multispectrum_formula_hypotheses.py",
    "prepare_test_guard5_candidates.py",
    "score_test_guard5_spectrum_fingerprint.py",
    "build_test_learned_ranker_features.py",
    "rank_test_learned_ranker.py",
    "rank_test_negative_retrieval.py",
    "build_submission.py",
]

missing_markers = [
    marker for marker in required_markers
    if marker not in orchestrator_text
]

if missing_markers:
    raise RuntimeError(
        "Attached inference asset is not the production baseline. "
        f"Missing stages: {missing_markers}"
    )

for forbidden in [
    "build_generated_test_formula_shortlist.py",
    "apply_generated_test_formula_hypotheses.py",
]:
    if forbidden in orchestrator_text:
        raise RuntimeError(
            "Attached inference asset still contains the regressed "
            f"generated-formula production path: {forbidden}"
        )

print("Preflight OK — learned-ranker production baseline")
print("RDKit wheel:", rdkit_wheels[0].name)
print("LightGBM wheel:", lightgbm_wheels[0].name)

Preflight OK
Orchestrator lines: 366
RDKit wheel: rdkit-2026.9.1-cp313-cp313-manylinux_2_28_x86_64.whl


## 2. Install and verify RDKit offline

This uses only the wheel packaged with the inference-assets dataset. It does not access PyPI.

In [3]:
rdkit_wheel = rdkit_wheels[0]

install = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--no-deps",
        "--force-reinstall",
        str(rdkit_wheel),
    ],
    text=True,
)

if install.returncode != 0:
    raise RuntimeError(
        f"Offline RDKit install failed with code {install.returncode}"
    )

verify = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import rdkit; "
            "from rdkit import Chem; "
            "assert Chem.MolFromSmiles('CCO') is not None; "
            "print('RDKit:', rdkit.__version__)"
        ),
    ],
    text=True,
)

if verify.returncode != 0:
    raise RuntimeError(
        f"RDKit verification failed with code {verify.returncode}"
    )

Processing /kaggle/input/datasets/colleendavis/casmi26-inference-assets/wheelhouse/rdkit-2026.9.1-cp313-cp313-manylinux_2_28_x86_64.whl
RDKit: 2026.09.1


## 3. Create a fresh writable project copy

Kaggle inputs are read-only. Every run starts from a clean copy in `/kaggle/working`.

In [4]:
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

shutil.copytree(
    ASSET_PROJECT,
    PROJECT_ROOT,
)

working_orchestrator = (
    PROJECT_ROOT
    / "scripts/run_kaggle_guard5_inference.py"
)

working_text = working_orchestrator.read_text()

expansion_root = (
    PROJECT_ROOT
    / "data/processed/pubchem_formula_index_runs/generated-test-top25"
)



Copied project to: /kaggle/working/casmi26-project
Working orchestrator lines: 366
Has wheelhouse setup: True
Has RDKit check: True


## 4. Run guard-5 inference

This is the expensive cell. Output is streamed directly so that any inner traceback remains visible.

If the HDF build is killed for memory, change `HDF_FORMULAS_PER_BATCH` to `2` and `HDF_WORKERS` to `1`, restart the notebook, and rerun from the top.

In [5]:
cmd = [
    sys.executable,
    "-u",
    str(
        PROJECT_ROOT
        / "scripts/run_kaggle_guard5_inference.py"
    ),
    "--project-root",
    str(PROJECT_ROOT),
    "--mist-root",
    str(MIST_ROOT),
    "--test-path",
    str(TEST_PATH),
    "--sample-submission-path",
    str(SAMPLE_SUBMISSION_PATH),
    "--hdf-formulas-per-batch",
    str(HDF_FORMULAS_PER_BATCH),
    "--hdf-workers",
    str(HDF_WORKERS),
]

print("Running:")
print(" ".join(cmd))
print()

result = subprocess.run(
    cmd,
    cwd=PROJECT_ROOT,
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(
        "Guard-5 inference failed. "
        f"Exit code: {result.returncode}. "
        "Use the first inner traceback printed above, not this wrapper error."
    )

Running:
/usr/bin/python3 -u /kaggle/working/casmi26-project/scripts/run_kaggle_guard5_inference.py --project-root /kaggle/working/casmi26-project --mist-root /kaggle/input/datasets/colleendavis/casmi26-inference-assets/MIST --test-path /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet --sample-submission-path /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv --hdf-formulas-per-batch 8 --hdf-workers 2


+ /usr/bin/python3 -m pip install --no-index --find-links /kaggle/input/datasets/colleendavis/casmi26-inference-assets/wheelhouse /kaggle/input/datasets/colleendavis/casmi26-inference-assets/wheelhouse/lightgbm-4.7.0-py3-none-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl /kaggle/input/datasets/colleendavis/casmi26-inference-assets/wheelhouse/rdkit-2026.9.1-cp313-cp313-manylinux_2_28_x86_64.whl
Looking in links: /kaggle/input/datasets/colleendavis/casmi26-inference-assets/wheelhouse
Processing /kaggle/input/datasets/c

## 5. Validate the generated submission

This checks row coverage, duplicate molecule IDs, missing predictions, and the maximum of 25 candidates per molecule before exposing the final file.

In [6]:
import pandas as pd

generated = PROJECT_ROOT / "submission.csv"

if not generated.exists():
    raise FileNotFoundError(
        f"Inference finished but {generated} does not exist"
    )

submission = pd.read_csv(generated)
sample = pd.read_csv(SAMPLE_SUBMISSION_PATH)

if list(submission.columns) != ["molecule_id", "smiles"]:
    raise ValueError(
        "Unexpected submission columns: "
        f"{list(submission.columns)}"
    )

if submission["molecule_id"].duplicated().any():
    raise ValueError("Submission has duplicate molecule_id values")

expected = sample["molecule_id"].astype(str)
actual = submission["molecule_id"].astype(str)

if len(submission) != len(sample):
    raise ValueError(
        f"Expected {len(sample)} rows, found {len(submission)}"
    )

if set(expected) != set(actual):
    missing = sorted(set(expected) - set(actual))
    extra = sorted(set(actual) - set(expected))
    raise ValueError(
        f"Molecule coverage mismatch. Missing={missing[:20]}, extra={extra[:20]}"
    )

if submission["smiles"].isna().any():
    raise ValueError("Submission contains missing smiles")

candidate_counts = (
    submission["smiles"]
    .astype(str)
    .str.split(";")
    .str.len()
)

if (candidate_counts < 1).any():
    raise ValueError("At least one molecule has no candidates")

if (candidate_counts > 25).any():
    raise ValueError("At least one molecule has more than 25 candidates")

print("Submission validation OK")
print("Rows:", len(submission))
print("Unique molecules:", submission["molecule_id"].nunique())
print()
print("Candidate-count distribution:")
print(candidate_counts.describe())
print()
print("Candidate-count frequencies:")
print(candidate_counts.value_counts().sort_index())

Submission validation OK
Rows: 400
Unique molecules: 400

Candidate-count distribution:
count    400.000000
mean      22.500000
std        6.296755
min        1.000000
25%       25.000000
50%       25.000000
75%       25.000000
max       25.000000
Name: smiles, dtype: float64

Candidate-count frequencies:
smiles
1       7
2       1
3       8
4       2
5       4
6       5
7       6
8       2
9       1
10      3
11      4
13      3
14      3
15      1
16      1
17      1
18      2
19      3
20      2
21      3
22      2
24      1
25    335
Name: count, dtype: int64


## 6. Publish the Kaggle output file

The final file is copied to `/kaggle/working/submission.csv`, which is the file to submit to the competition.

In [7]:
FINAL_SUBMISSION = Path("/kaggle/working/submission.csv")

shutil.copy2(
    generated,
    FINAL_SUBMISSION,
)

print("Final submission:", FINAL_SUBMISSION)
print("Size:", FINAL_SUBMISSION.stat().st_size, "bytes")
print()
display(
    pd.read_csv(FINAL_SUBMISSION).head()
)

Final submission: /kaggle/working/submission.csv
Size: 355447 bytes



,molecule_id,smiles
0,m_005e53,Cn1ccnc1C(O)(CCNc1ncc(C(F)(F)F)s1)C(F)(F)F;O=C...
1,m_006153,COc1ccc(N2CC(C(=O)Nc3nncn3C3CC3)CC2=O)cc1;CC(C...
2,m_00b5aa,COc1ccnc(CN2CCC3(CCOCC3)C2C)c1OC;COc1ncccc1CN1...
3,m_01e922,CCOC(=O)Nc1cccc(OC2CCCC2)c1;CCCN(CC(=O)O)C(=O)...
4,m_0259d4,CC(=O)N1CCN(c2ncc(F)c(N3CCOCC3)n2)CC(O)C1;CN(C...
